<a href="https://colab.research.google.com/github/Sunanda3030/Anomaly-Detection-for-Fraud-and-Sensor-Data/blob/main/12_Anomaly_Detection_Fraud_Sensor_Data_Summer_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Anomaly Detection for Fraud and Sensor Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Rounak Biswas<br>
  <strong>Designation:</strong> Project Linked Associate Research Engineer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Load Libraries (2 Marks)

Import `numpy` as `np`, `pandas` as `pd`, `stats` from `scipy`, `IsolationForest` and `LocalOutlierFactor` from `sklearn.ensemble` and `sklearn.neighbors`, and `classification_report`, `precision_score`, `recall_score` from `sklearn.metrics`.

**Expected Output:** The code cell should execute without any errors.

In [ ]:
import numpy as np
import pandas as pd

from scipy import stats

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

### Question 2: Create the Dataset (4 Marks)

Generate a synthetic credit card transaction dataset. Set `np.random.seed(42)`. Create a DataFrame `normal` with 950 rows and a DataFrame `fraud` with 50 rows. Features should include `amount`, `hour_of_day`, `transactions_last_24h`, and `distance_from_home_km`, plus an `is_fraud` label (0 for normal, 1 for fraud). Combine them into a single DataFrame named `df`, shuffle using `.sample(frac=1, random_state=42)`, and reset the index.

**Hint:** Use `np.random.normal`, `np.random.randint`, `np.random.poisson`, and `np.random.exponential` to generate feature data.

**Expected Output:** Execution without errors, creating the `df` DataFrame.

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score
from google.colab import files
uploaded = files.upload()
df = pd.read_excel("EPC_Industry_Analytics_Demo.xlsx")
print(df.head())
# Set random seed
np.random.seed(42)

# Create normal transactions (950 rows)
normal = pd.DataFrame({
    'amount': np.random.normal(loc=50, scale=15, size=950),
    'hour_of_day': np.random.randint(0, 24, 950),
    'transactions_last_24h': np.random.poisson(lam=3, size=950),
    'distance_from_home_km': np.random.exponential(scale=5, size=950),
    'is_fraud': 0
})

# Create fraudulent transactions (50 rows)
fraud = pd.DataFrame({
    'amount': np.random.normal(loc=250, scale=80, size=50),
    'hour_of_day': np.random.randint(0, 24, 50),
    'transactions_last_24h': np.random.poisson(lam=12, size=50),
    'distance_from_home_km': np.random.exponential(scale=30, size=50),
    'is_fraud': 1
})

# Combine datasets
df = pd.concat([normal, fraud], ignore_index=True)

# Shuffle and reset index
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Display first few rows
print(df.head())

Saving EPC_Industry_Analytics_Demo.xlsx to EPC_Industry_Analytics_Demo (13).xlsx
        Date Project_ID         Region      Project_Type Contractor_Tier  \
0 2023-01-01  PROJ-1000  North America  Renewable Energy          Tier 3
1 2023-01-01  PROJ-1001          LATAM  Industrial Plant          Tier 1
2 2023-01-01  PROJ-1002           APAC         Oil & Gas          Tier 1
3 2023-01-01  PROJ-1003  North America  Industrial Plant          Tier 2
4 2023-01-02  PROJ-1004  North America  Renewable Energy          Tier 2

   Planned_Budget_K  Actual_Cost_K  Budget_Variance_Pct  \
0       2032.718750    2095.845956             3.105555
1        822.703500     839.771030             2.074566
2       2343.330027    2501.768017             6.761232
3       1900.478728    1966.592256             3.478783
4       3547.042879    3508.019596            -1.100164

   Sensor_Vibration_mm_s  Sensor_Temp_C  Efficiency_Score  \
0               4.775256      63.900729         64.144031
1               1.779238      46.502858         83.031245
2               2.303336      50.985558         81.513558
3               3.341599      59.448200         72.523566
4               3.501433      59.905765         70.703579

   Material_Price_Index  Unnamed: 12 Unnamed: 13
0            100.000000          NaN         NaN
1            100.058382          NaN         NaN
2            100.116760          NaN         NaN
3            100.175133          NaN         NaN
4            100.233497          NaN         NaN
      amount  hour_of_day  transactions_last_24h  distance_from_home_km  \
0  58.150403           17                      1               3.400957
1  64.740365           10                      2               6.331045
2  22.386887           18                      0               0.275963
3  41.395070            0                      0               2.323902
4  33.130369            1                      3               3.217447
is_fraud
0         0
1         0
2         0
3         0
4         0

### Question 3: Check Dataset Shape and Distribution (2 Marks)

Print the shape of your combined DataFrame `df` and the value counts of the `is_fraud` column to observe the class distribution.

**Expected Output:** The shape (1000, 5) and the counts showing 950 normal (0) and 50 fraud (1) cases.

In [ ]:
# Display dataset shape
print("Dataset Shape:", df.shape)

# Display class distribution
print("\nClass Distribution:")
print(df['is_fraud'].value_counts())

# Print dataset shape
print("Dataset Shape:", df.shape)

# Print class distribution
print("\nClass Distribution:")
print(df['is_fraud'].value_counts())

Dataset Shape: (1000, 5)

### Question 4: Compare Feature Means by Class (3 Marks)

Group the dataset by the `is_fraud` column and calculate the mean values for all features (`amount`, `hour_of_day`, `transactions_last_24h`, `distance_from_home_km`). Print the resulting grouped means.

**Hint:** Use the `.groupby()` method and `.mean()`.

**Expected Output:** A table showing the mean feature values for normal (0) vs fraudulent (1) transactions.

In [ ]:
 # Print class distribution
print("\nClass Distribution:")
print(df['is_fraud'].value_counts())
# Calculate mean values of features grouped by fraud class
feature_means = df.groupby('is_fraud')[[
    'amount',
    'hour_of_day',
    'transactions_last_24h',
    'distance_from_home_km'
]].mean()

# Print the grouped means
print(feature_means)

Class Distribution:
is_fraud
0    950
1     50
Name: count, dtype: int64

amount  hour_of_day  transactions_last_24h  \
is_fraud
0          50.301261    11.394737               2.984211
1         251.404503    12.080000              11.580000

          distance_from_home_km
is_fraud
0                      5.035938
1                     32.579013

### Question 5: Apply Z-Score Anomaly Detection (3 Marks)

Compute the absolute Z-scores for the `distance_from_home_km` column. Create a new column named `zscore_anomaly` in `df` that contains `1` if the absolute Z-score is greater than 3, and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `np.abs(stats.zscore(...))`.

**Expected Output:** The count of anomalies flagged based on the distance feature.

In [ ]:
# Compute absolute Z-scores for distance_from_home_km
z_scores = np.abs(stats.zscore(df['distance_from_home_km']))

# Create anomaly flag column
df['zscore_anomaly'] = (z_scores > 3).astype(int)

# Print total number of anomalies
print("Total Z-Score Anomalies:", df['zscore_anomaly'].sum())

Total Z-Score Anomalies: 20

### Question 6: Apply IQR Method (4 Marks)

Calculate the Interquartile Range (IQR) for the `amount` column. Identify bounds: `Lower = Q1 - 1.5 * IQR` and `Upper = Q3 + 1.5 * IQR`. Create a new column named `iqr_anomaly` in `df` containing `1` for values outside these bounds and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `.quantile(0.25)` for Q1 and `.quantile(0.75)` for Q3.

**Expected Output:** The total number of `amount` anomalies flagged by the IQR method.

In [ ]:
# Calculate Q1 and Q3
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Define lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

Total IQR Anomalies: 54

### Question 7: Train Isolation Forest (4 Marks)

Import `StandardScaler` from `sklearn.preprocessing` and scale the four feature columns. Then, create an `IsolationForest` model with `contamination=0.05` and `random_state=42`. Fit the model on the scaled features and add a column `isoforest_anomaly` to `df` containing `1` for anomalies and `0` for normal data.

**Hint:** `IsolationForest` returns `-1` for anomalies and `1` for normal points. Map these to `1` and `0` respectively.

**Expected Output:** The execution completes successfully.

In [ ]:
# Import StandardScaler
from sklearn.preprocessing import StandardScaler

# Select feature columns
features = [
    'amount',
    'hour_of_day',
    'transactions_last_24h',
    'distance_from_home_km'
]
# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

Total Isolation Forest Anomalies:50

### Question 8: Evaluate Isolation Forest (3 Marks)

Use the `classification_report` function to evaluate the performance of your `isoforest_anomaly` predictions against the true `is_fraud` labels. Print the report.

**Expected Output:** A classification report displaying precision, recall, and f1-score for the model.

In [ ]:
# Evaluate Isolation Forest predictions
report = classification_report(
    df['is_fraud'],
    df['isoforest_anomaly']
)

# Print the classification report
print(report)

precision    recall  f1-score   support

           0       1.00      1.00      1.00       950
           1       0.96      0.96      0.96        50

    accuracy                           1.00      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       1.00      1.00      1.00      1000

### Question 9: Local Outlier Factor Detection (3 Marks)

Train a `LocalOutlierFactor` model with `n_neighbors=20` and `contamination=0.05` on the scaled features. Add a new column `lof_anomaly` to `df` (where `1` indicates an anomaly and `0` indicates normal).

**Hint:** Use `.fit_predict()` to get the anomaly flags (similar to Isolation Forest, LOF returns `-1` for anomalies).

**Expected Output:** The execution completes successfully.

In [ ]:
# Create and train Local Outlier Factor model
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)
# Predict anomalies
lof_predictions = lof.fit_predict(X_scaled)
# Convert predictions:
# -1 -> 1 (anomaly)
#  1 -> 0 (normal)
df['lof_anomaly'] = np.where(lof_predictions == -1, 1, 0)
# Print total anomalies detected
print("Total LOF Anomalies:", df['lof_anomaly'].sum())
Total LOF Anomalies: 50

### Question 10: Evaluate LOF Model (2 Marks)

Calculate and print both the `precision_score` and `recall_score` for the `lof_anomaly` predictions against the true `is_fraud` labels.

**Expected Output:** Two numbers showing the precision and recall scores.

In [ ]:
 # Calculate precision and recall for LOF predictions
precision = precision_score(
    df['is_fraud'],
    df['lof_anomaly']
)

recall = recall_score(
    df['is_fraud'],
    df['lof_anomaly']
)

# Print results
print("LOF Precision:", precision)
print("LOF Recall:", recall)
LOF Precision: 0.26
LOF Recall: 0.26